# Tutorial 4: Custom Shapefiles for Freshwater Organisms

This tutorial demonstrates how to use `boldgenotyper-enrich` with custom shapefiles to analyze freshwater organisms using drainage basin classifications.

## Learning Objectives

By the end of this tutorial, you will:
- Understand when and why to use custom shapefiles
- Learn to inspect shapefile structure and attributes
- Run enrichment analysis with freshwater basin shapefiles
- Analyze genotype distributions across drainage basins
- Extend BOLDGenotyper to terrestrial organisms

## Prerequisites

- Completed Tutorial 1 (Basic Genotyping Workflow)
- Freshwater dataset: `Salmonidae.tsv` or `Cyprinodontidae.tsv`
- Custom shapefile (e.g., HydroBASINS from https://www.hydrosheds.org/)
- Basic understanding of GIS concepts

## Why Custom Shapefiles?

BOLDGenotyper's default geographic assignment uses **ocean basins** (GOaS), which is appropriate for marine organisms. For freshwater and terrestrial organisms, you need different geographic classifications:

- **Freshwater**: Drainage basins, watersheds, river systems
- **Terrestrial**: Ecoregions, biomes, biogeographic realms
- **Custom**: Study areas, conservation zones, political boundaries

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import geopandas as gpd
from shapely.geometry import Point

## Step 1: Obtain Custom Shapefile

For this tutorial, we'll use HydroBASINS Level 4 for drainage basin classification.

### Download HydroBASINS:
1. Visit: https://www.hydrosheds.org/products/hydrobasins
2. Download global or regional level 4 basins
3. Extract to `data/shapefiles/hydrobasins_lev04/`

Note: This tutorial assumes you've already downloaded the shapefile.

## Step 2: Inspect Shapefile Structure

Before using a shapefile, inspect its structure and attributes.

In [ ]:
# Path to your shapefile (update this path)
shapefile_path = "../data/shapefiles/hydrobasins_lev04/hybas_lev04.shp"

# Check if shapefile exists
if Path(shapefile_path).exists():
    print(f"Shapefile found: {shapefile_path}")
    
    # Load shapefile
    gdf = gpd.read_file(shapefile_path)
    
    print(f"\nShapefile info:")
    print(f"  Features: {len(gdf)}")
    print(f"  CRS: {gdf.crs}")
    print(f"  Bounds: {gdf.total_bounds}")
    
    print(f"\nAttribute fields:")
    print(gdf.columns.tolist())
    
    print(f"\nFirst few features:")
    print(gdf.head())
else:
    print(f"Shapefile not found at: {shapefile_path}")
    print("Please download HydroBASINS from https://www.hydrosheds.org/")
    print("This tutorial will continue with example code.")

In [ ]:
# Visualize shapefile (if available)
if Path(shapefile_path).exists():
    fig, ax = plt.subplots(figsize=(12, 8))
    gdf.plot(ax=ax, edgecolor='black', facecolor='lightblue', linewidth=0.5)
    ax.set_title('HydroBASINS Level 4 - Global Drainage Basins')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()
    
    # Show attribute distribution
    if 'MAIN_BAS' in gdf.columns:
        print("\nMajor basin ID distribution:")
        print(gdf['MAIN_BAS'].value_counts().head(10))

## Step 3: Run Initial Genotyping Without Geography

First, run the basic genotyping analysis without geographic assignment.

In [ ]:
# Run genotyping without geographic assignment
# Use --no-geo to skip ocean basin assignment

!boldgenotyper ../data/Salmonidae.tsv \
  --no-geo \
  --clustering-threshold 0.02 \
  --output ../data/Salmonidae_nogeo/ \
  --threads 4

## Step 4: Inspect Genotyping Results

In [ ]:
# Load genotyping results
results = pd.read_csv("../data/Salmonidae_nogeo/Salmonidae_annotated.csv")

print(f"Dataset: Salmonidae (salmon and trout)")
print(f"Total samples: {len(results)}")
print(f"Total genotypes: {results['genotype'].nunique()}")
print(f"\nSamples with coordinates: {results['lat'].notna().sum()}")
print(f"Species represented: {results['species_name'].nunique()}")

print("\nTop species:")
print(results['species_name'].value_counts().head(10))

## Step 5: Run Enrichment with Custom Shapefile

Now use `boldgenotyper-enrich` to add drainage basin information.

In [ ]:
# Run enrichment with HydroBASINS shapefile
# Key parameters:
#   --custom-shp: Path to shapefile
#   --shp-field: Attribute field to use for classification
#   --geo-category: Name for the new geographic category

!boldgenotyper-enrich ../data/Salmonidae_nogeo/Salmonidae_annotated.csv \
  --custom-shp ../data/shapefiles/hydrobasins_lev04/hybas_lev04.shp \
  --shp-field MAIN_BAS \
  --geo-category drainage_basin \
  --output ../data/Salmonidae_enriched/

## Step 6: Examine Enriched Results

In [ ]:
# Load enriched results
enriched = pd.read_csv("../data/Salmonidae_enriched/Salmonidae_enriched.csv")

print("Enriched dataset:")
print(f"  Total samples: {len(enriched)}")
print(f"  New columns: {[col for col in enriched.columns if col not in results.columns]}")

# Check drainage basin assignment
if 'drainage_basin' in enriched.columns:
    assigned = enriched['drainage_basin'].notna().sum()
    print(f"\nDrainage basin assignment:")
    print(f"  Samples assigned: {assigned} ({assigned/len(enriched)*100:.1f}%)")
    print(f"  Unique basins: {enriched['drainage_basin'].nunique()}")
    
    print(f"\nTop drainage basins:")
    print(enriched['drainage_basin'].value_counts().head(10))

## Step 7: Analyze Genotype Distribution by Drainage Basin

In [ ]:
# Cross-tabulation of genotypes by drainage basin
if 'drainage_basin' in enriched.columns:
    # Filter to assigned samples
    assigned_samples = enriched[enriched['drainage_basin'].notna()].copy()
    
    # Create crosstab
    basin_genotype = pd.crosstab(assigned_samples['drainage_basin'], 
                                  assigned_samples['genotype'])
    
    print(f"\nGenotype-Basin Crosstab:")
    print(f"  Basins: {basin_genotype.shape[0]}")
    print(f"  Genotypes: {basin_genotype.shape[1]}")
    
    # Top basins by sample count
    basin_totals = basin_genotype.sum(axis=1).sort_values(ascending=False)
    print(f"\nTop 10 basins by sample count:")
    print(basin_totals.head(10))

In [ ]:
# Visualize basin distribution
if 'drainage_basin' in enriched.columns and len(basin_totals) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Basin sample counts
    basin_totals.head(15).plot(kind='barh', ax=axes[0])
    axes[0].set_xlabel('Number of Samples')
    axes[0].set_ylabel('Drainage Basin ID')
    axes[0].set_title('Sample Distribution Across Top 15 Drainage Basins')
    
    # Genotypes per basin
    genotypes_per_basin = (basin_genotype > 0).sum(axis=1).sort_values(ascending=False)
    genotypes_per_basin.head(15).plot(kind='barh', ax=axes[1], color='orange')
    axes[1].set_xlabel('Number of Genotypes')
    axes[1].set_ylabel('Drainage Basin ID')
    axes[1].set_title('Genotype Diversity per Basin (Top 15)')
    
    plt.tight_layout()
    plt.show()

## Step 8: Identify Basin-Specific Genotypes

In [ ]:
# Find genotypes restricted to single basins (potential endemic genotypes)
if 'drainage_basin' in enriched.columns:
    # Count basins per genotype
    basins_per_genotype = assigned_samples.groupby('genotype')['drainage_basin'].nunique()
    
    # Endemic genotypes (only in one basin)
    endemic = basins_per_genotype[basins_per_genotype == 1]
    
    # Widespread genotypes (in 5+ basins)
    widespread = basins_per_genotype[basins_per_genotype >= 5]
    
    print("GENOTYPE DISTRIBUTION PATTERNS")
    print("="*80)
    print(f"\nEndemic genotypes (1 basin only): {len(endemic)}")
    print(f"  Example: {endemic.head(5).to_dict()}")
    
    print(f"\nWidespread genotypes (5+ basins): {len(widespread)}")
    if len(widespread) > 0:
        print(f"  Most widespread: {basins_per_genotype.idxmax()} ({basins_per_genotype.max()} basins)")
    
    print(f"\nBasin distribution:")
    print(f"  Mean basins per genotype: {basins_per_genotype.mean():.1f}")
    print(f"  Median basins per genotype: {basins_per_genotype.median():.1f}")
    print("="*80)

## Step 9: Geographic Visualization

In [ ]:
# Create map of sample locations colored by drainage basin
if 'drainage_basin' in enriched.columns and 'lat' in enriched.columns:
    # Filter to samples with coordinates and basin assignment
    map_data = enriched[(enriched['lat'].notna()) & 
                        (enriched['lon'].notna()) & 
                        (enriched['drainage_basin'].notna())].copy()
    
    # Get top 10 basins for coloring
    top_basins = map_data['drainage_basin'].value_counts().head(10).index
    map_data['basin_category'] = map_data['drainage_basin'].apply(
        lambda x: str(x) if x in top_basins else 'Other'
    )
    
    # Create map
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Plot each basin category
    for basin in map_data['basin_category'].unique():
        subset = map_data[map_data['basin_category'] == basin]
        ax.scatter(subset['lon'], subset['lat'], 
                  label=f'Basin {basin}', alpha=0.6, s=20)
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Sample Locations by Drainage Basin')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Step 10: Generate Summary Statistics

In [ ]:
# Create comprehensive summary
if 'drainage_basin' in enriched.columns:
    summary_stats = {
        'total_samples': len(enriched),
        'assigned_samples': enriched['drainage_basin'].notna().sum(),
        'assignment_rate': enriched['drainage_basin'].notna().sum() / len(enriched) * 100,
        'unique_basins': enriched['drainage_basin'].nunique(),
        'total_genotypes': enriched['genotype'].nunique(),
        'endemic_genotypes': len(endemic),
        'widespread_genotypes': len(widespread),
    }
    
    print("ENRICHMENT SUMMARY STATISTICS")
    print("="*80)
    for key, value in summary_stats.items():
        if 'rate' in key or 'percent' in key:
            print(f"{key.replace('_', ' ').title()}: {value:.1f}%")
        else:
            print(f"{key.replace('_', ' ').title()}: {value}")
    print("="*80)
    
    # Save summary
    summary_df = pd.DataFrame([summary_stats])
    summary_df.to_csv("../data/Salmonidae_enriched/enrichment_summary.csv", index=False)
    print("\nSummary saved to: ../data/Salmonidae_enriched/enrichment_summary.csv")

## Step 11: Generate Publication Methods Text

In [ ]:
# Create methods text for publication
methods_text = f"""
METHODS - Geographic Assignment Using Custom Shapefiles

Geographic classification was performed using custom drainage basin boundaries 
rather than ocean basins, as appropriate for freshwater organisms. We used 
HydroBASINS Level 4 (Lehner & Grill, 2013) to assign samples to major drainage 
basins based on collection coordinates.

The enrichment analysis was conducted using boldgenotyper-enrich (v0.1.0), which 
performed spatial joins between sample coordinates and basin polygons. Of 
{len(enriched)} total samples, {enriched['drainage_basin'].notna().sum()} 
({enriched['drainage_basin'].notna().sum()/len(enriched)*100:.1f}%) were 
successfully assigned to drainage basins.

The analysis identified {enriched['drainage_basin'].nunique()} unique drainage 
basins represented in the dataset. Genotype distributions across basins revealed 
{len(endemic) if 'endemic' in locals() else 'N/A'} basin-endemic genotypes and 
{len(widespread) if 'widespread' in locals() else 'N/A'} widespread genotypes 
occurring in 5 or more basins.

CITATION:
Lehner, B., & Grill, G. (2013). Global river hydrography and network routing: 
baseline data and new approaches to study the world's large river systems. 
Hydrological Processes, 27(15), 2171-2186.
"""

print(methods_text)

# Save methods text
with open("../data/Salmonidae_enriched/methods_text.txt", 'w') as f:
    f.write(methods_text)
    
print("\nMethods text saved to: ../data/Salmonidae_enriched/methods_text.txt")

## Key Takeaways

1. **Universal Application**: BOLDGenotyper works for any organism type with custom shapefiles
2. **Biologically Relevant**: Use shapefiles that match organism ecology (drainage basins for freshwater, ecoregions for terrestrial)
3. **Flexible Classification**: Any polygon shapefile with attribute fields can be used
4. **Publication-Ready**: Automated methods text generation with proper citations
5. **Conservation Applications**: Identify endemic genotypes, priority basins, dispersal patterns

## Other Use Cases

### Terrestrial Organisms
```python
# Use WWF terrestrial ecoregions
!boldgenotyper-enrich butterfly_annotated.csv \
  --custom-shp wwf_ecoregions.shp \
  --shp-field ECO_NAME \
  --geo-category ecoregion
```

### Conservation Areas
```python
# Use protected area boundaries
!boldgenotyper-enrich species_annotated.csv \
  --custom-shp protected_areas.shp \
  --shp-field PA_NAME \
  --geo-category protected_area
```

### Custom Study Areas
```python
# Use your own study site polygons
!boldgenotyper-enrich samples_annotated.csv \
  --custom-shp study_sites.shp \
  --shp-field site_id \
  --geo-category study_site
```

## Best Practices

1. **Inspect shapefiles first** - Understand attribute fields and CRS
2. **Choose appropriate scale** - Match resolution to your question
3. **Document sources** - Cite shapefile providers properly
4. **Validate assignments** - Check that assignments make biological sense
5. **Consider hierarchies** - Use nested classifications (e.g., Level 4 and Level 6 basins)

## Shapefile Resources

1. **HydroBASINS**: https://www.hydrosheds.org/ (freshwater drainage basins)
2. **FEOW**: https://www.feow.org/ (freshwater ecoregions)
3. **WWF Ecoregions**: https://www.worldwildlife.org/publications/terrestrial-ecoregions-of-the-world (terrestrial ecoregions)
4. **WDPA**: https://www.protectedplanet.net/ (protected areas)
5. **Natural Earth**: https://www.naturalearthdata.com/ (general GIS data)

## Next Steps

- **Tutorial 5**: Export for population genetics analysis
- **Custom Shapefiles Guide**: `../CUSTOM_SHAPEFILES_GUIDE.md`
- **Advanced usage**: Combining multiple shapefiles, creating custom shapefiles

## Additional Resources

- QGIS tutorials for shapefile creation and manipulation
- GeoPandas documentation for Python-based GIS
- BOLDGenotyper API for programmatic shapefile integration